# Ridge Regression (L2 Regularization)

In linear regression, standard OLS optimization strives to minimize the Mean Squared Error over the training set. However, if a dataset has many features or exhibits high multicollinearity, OLS can result in very large coefficients. This makes the model extremely sensitive to minor fluctuations in the data, leading to **overfitting**. **Regularization** adds a constraint (or penalty) to the cost function to shrink these coefficients, trading a small amount of bias for a massive reduction in variance.

---

## 1. Introduction to Regularization

Regularization is a set of techniques used to introduce structural constraints to a machine learning model to prevent overfitting and improve generalization on test data.

There are three primary types of regularization for linear models:

| Regularization Type | Mathematical Penalty | Behavior on Coefficients | Use Case |
| :--- | :--- | :--- | :--- |
| **Ridge Regression (L2)** | Squared magnitude of coefficients: $\lambda \sum \beta_j^2$ | Shrinks coefficients **near zero** (never exactly zero) | Mitigates multicollinearity; keeps all features |
| **Lasso Regression (L1)** | Absolute magnitude of coefficients: $\lambda \sum |\beta_j|$ | Shrinks coefficients **exactly to zero** | Performs automatic feature selection; sparse models |
| **Elastic Net** | Combined L1 and L2 penalty | Hybrid shrinkage | Useful when features are highly correlated in groups |

---

## 2. Understanding Overfitting and Coefficient Size

Why do large coefficients cause overfitting?

Let's look at a simple prediction equation: $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2$.
- If $\beta_1 = 1.2$, a 1-unit increase in $x_1$ increases predictions by $1.2$.
- If $\beta_1 = 850.0$, a tiny 0.01-unit noise or fluctuation in $x_1$ shifts predictions by $8.5$! 

Large coefficient values indicate that the model is overly sensitive to the training inputs. Regularization addresses this by penalizing large coefficient values, forcing the optimization algorithm to find smaller, more stable weights.

## 3. The Mathematics of Ridge Regression

Ridge Regression modifies the standard OLS loss function by adding a penalty proportional to the sum of the squared coefficients (excluding the intercept $\beta_0$):

$$L_{\text{Ridge}}(\beta) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{m} \beta_j^2$$

Where:
- **$\lambda$ (alpha in Scikit-Learn):** A non-negative hyperparameter controlling the strength of regularization.
  - If **$\lambda = 0$**, Ridge becomes standard OLS Linear Regression (high risk of overfitting).
  - If **$\lambda \rightarrow \infty$**, the penalty dominates, forcing all coefficients $\beta_j \rightarrow 0$ (high risk of underfitting, intercept $\beta_0$ remains standard average).
- **$\sum_{j=1}^{m} \beta_j^2$:** The L2 penalty. Notice it starts at $j=1$ (coefficients) and does **not** penalize the intercept term $\beta_0$. There is no benefit in forcing the baseline average $\beta_0$ to zero.

### 3.1 Mathematical Derivation: 1D Case

To understand the effect of the L2 penalty, let's derive the analytical solution for a 1D model with centered data (meaning intercept $b=0$). The model is $\hat{y}_i = m x_i$.

Our cost function is:

$$E(m) = \sum_{i=1}^{n} (y_i - m x_i)^2 + \lambda m^2$$

To find the slope $m$ that minimizes this cost, we differentiate with respect to $m$ and set it to $0$:

$$\frac{dE}{dm} = \sum_{i=1}^{n} 2(y_i - m x_i)(-x_i) + 2\lambda m = 0$$

$$-2 \sum_{i=1}^{n} (x_i y_i - m x_i^2) + 2\lambda m = 0$$

$$-\sum_{i=1}^{n} x_i y_i + m \sum_{i=1}^{n} x_i^2 + \lambda m = 0$$

$$m \left( \sum_{i=1}^{n} x_i^2 + \lambda \right) = \sum_{i=1}^{n} x_i y_i$$

$$m = \frac{\sum_{i=1}^{n} x_i y_i}{\sum_{i=1}^{n} x_i^2 + \lambda}$$

Using mean-centered values ($x_i - \bar{x}$) and ($y_i - \bar{y}$), we get the slope formula:

$$m = \frac{\sum_{i=1}^{n} (y_i - \bar{y})(x_i - \bar{x})}{\sum_{i=1}^{n} (x_i - \bar{x})^2 + \lambda}$$

**Conclusion:** The OLS slope is divided by a larger denominator ($+ \lambda$). As we increase $\lambda$, the denominator grows, which mathematically forces the slope $m$ to shrink closer to $0$.

### 3.2 N-Dimensional Closed-Form Formulation

For multiple features, we write the cost function in matrix form:

$$L(\beta) = (Y - X\beta)^T (Y - X\beta) + \lambda (I^* \beta)^T (I^* \beta)$$

Where $I^*$ is the **modified identity matrix** of size $(m+1) \times (m+1)$:

$$I^* = \begin{bmatrix} 
0 & 0 & 0 & \dots & 0 \\ 
0 & 1 & 0 & \dots & 0 \\ 
0 & 0 & 1 & \dots & 0 \\ 
\vdots & \vdots & \vdots & \ddots & \vdots \\ 
0 & 0 & 0 & \dots & 1 
\end{bmatrix}$$

Setting the first element (index $0,0$) to $0$ ensures that we do **not** penalize the intercept term $\beta_0$.

Taking the derivative with respect to the vector $\beta$:

$$\nabla L(\beta) = -2 X^T (Y - X\beta) + 2\lambda I^* \beta = 0$$

$$-X^T Y + X^T X \beta + \lambda I^* \beta = 0$$

$$(X^T X + \lambda I^*) \beta = X^T Y$$

$$\beta = (X^T X + \lambda I^*)^{-1} X^T Y$$

This is the **analytical closed-form solution** for Ridge Regression. Adding $\lambda$ along the diagonal makes the matrix $(X^T X + \lambda I^*)$ mathematically invertible even if features are highly collinear, resolving standard OLS matrix singularity issues.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Generate a synthetic dataset with multicollinearity (150 samples, 8 features)
np.random.seed(42)
X, y = make_regression(n_samples=150, n_features=8, noise=15, random_state=42)

# Induce severe multicollinearity: make features 6 and 7 copies of feature 0 and 1 with tiny noise
X[:, 5] = X[:, 0] * 0.95 + np.random.normal(0, 0.05, 150)
X[:, 6] = X[:, 1] * 0.98 + np.random.normal(0, 0.03, 150)

# Scale the features (Regularization is highly scale-sensitive)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print("Multicollinearity generated. Features 5 and 6 are highly correlated with features 0 and 1.")


## 4. Custom Analytical Implementation (Closed-Form)

Let's build a custom class `MyRidgeOLS` from scratch that implements the analytical matrix solution $\beta = (X^T X + \lambda I^*)^{-1} X^T Y$, ensuring we do not penalize the intercept term.

In [ ]:
class MyRidgeOLS:
    """Custom Analytical closed-form Ridge Regression solver"""
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # alpha is lambda in our formulas
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # 1. Insert a column of 1s at index 0 to account for intercept
        X_new = np.insert(X_train, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        
        # 2. Create the modified Identity Matrix (I*)
        I_star = np.identity(n_features)
        I_star[0, 0] = 0.0  # Do not penalize the intercept!
        
        # 3. Solve the closed form: beta = (X^T X + alpha * I^*)^-1 * X^T * Y
        A = np.dot(X_new.T, X_new) + self.alpha * I_star
        B = np.dot(X_new.T, y_train)
        beta = np.dot(np.linalg.inv(A), B)
        
        # 4. Extract intercept and coefficients
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_


In [ ]:
# Fit custom Ridge OLS
my_ridge = MyRidgeOLS(alpha=10.0)
my_ridge.fit(X_train, y_train)

# Fit Scikit-Learn Ridge OLS
sk_ridge = Ridge(alpha=10.0, solver='cholesky')
sk_ridge.fit(X_train, y_train)

# Compare outputs
print("="*55)
print("CLOSED-FORM ANALYTICAL RESULTS COMPARISON (alpha=10.0)")
print("="*55)
print(f"Custom Intercept:   {my_ridge.intercept_:.6f}")
print(f"Sklearn Intercept:  {sk_ridge.intercept_:.6f}")
print(f"Custom Coefficients:\n{my_ridge.coef_}")
print(f"Sklearn Coefficients:\n{sk_ridge.coef_}")
print(f"\nTest set R2 Score (Custom):  {r2_score(y_test, my_ridge.predict(X_test)):.6f}")
print(f"Test set R2 Score (Sklearn): {r2_score(y_test, sk_ridge.predict(X_test)):.6f}")
print("="*55)


## 5. Ridge Regression via Gradient Descent

For extremely large datasets, computing the inverse of $X^T X$ is computationally expensive ($O(m^3)$ matrix operations). In these scenarios, we use **Gradient Descent** to iteratively search for parameters.

### 5.1 Derivative of the Regularized Loss
Differentiating the Ridge Loss function with respect to the weight vector $\beta$:

$$\nabla L(\beta) = -\frac{2}{n} X^T (Y - X\beta) + 2\lambda I^* \beta$$

Separated for clarity:
- **For Intercept ($\beta_0$ - No penalty):**
  $$\frac{\partial L}{\partial \beta_0} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$
- **For Coefficient ($\beta_j$ - Regularized):**
  $$\frac{\partial L}{\partial \beta_j} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) x_{ij} + 2\lambda \beta_j$$

### 5.2 Gradient Descent Update Rule
At each iteration, parameters are updated simultaneously using learning rate $\eta$:

$$\beta_{\text{new}} = \beta_{\text{old}} - \eta \cdot \left[ -\frac{2}{n} X^T (Y - X\beta_{\text{old}}) + 2\lambda I^* \beta_{\text{old}} \right]$$

$$\beta_{\text{new}} = \beta_{\text{old}} + \frac{2\eta}{n} X^T (Y - X\beta_{\text{old}}) - 2\eta \lambda I^* \beta_{\text{old}}$$

**Intuition of weight decay:** Notice the third term $-2\eta \lambda I^* \beta_{\text{old}}$. At each step, this term multiplies the current coefficient vector by a factor of $(1 - 2\eta\lambda)$, shrinking the coefficient values slightly before adding the standard OLS gradient step. This is known as **Weight Decay**.

In [ ]:
class MyRidgeGD:
    """Custom Gradient Descent Ridge Regression solver"""
    def __init__(self, alpha=1.0, learning_rate=0.01, epochs=1000):
        self.alpha = alpha
        self.lr = learning_rate
        self.epochs = epochs
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # Add column of 1s to input
        X_new = np.insert(X_train, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        
        # Initialize beta vector to zeros
        beta = np.zeros(n_features)
        
        # Modified Identity Matrix to ignore intercept penalty
        I_star = np.identity(n_features)
        I_star[0, 0] = 0.0
        
        # Gradient descent loop
        for epoch in range(self.epochs):
            y_pred = np.dot(X_new, beta)
            
            # Gradient: d_beta = -2/n * X^T * (y - y_pred) + 2 * alpha * I^* * beta
            d_beta = - (2 / n_samples) * np.dot(X_new.T, y_train - y_pred) + 2 * self.alpha * np.dot(I_star, beta)
            
            # Parameter Update
            beta = beta - self.lr * d_beta
            
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_


In [ ]:
# Fit custom Gradient Descent model
my_ridge_gd = MyRidgeGD(alpha=10.0, learning_rate=0.05, epochs=1500)
my_ridge_gd.fit(X_train, y_train)

print("="*55)
print("GRADIENT DESCENT RESULTS COMPARISON (alpha=10.0)")
print("="*55)
print(f"GD Intercept:        {my_ridge_gd.intercept_:.6f}")
print(f"Sklearn Intercept:   {sk_ridge.intercept_:.6f}")
print(f"GD Coefficients:\n{my_ridge_gd.coef_}")
print(f"Sklearn Coefficients:\n{sk_ridge.coef_}")
print(f"\nGD Test set R2 Score: {r2_score(y_test, my_ridge_gd.predict(X_test)):.6f}")
print("="*55)


## 6. Hyperparameter Tuning & Coefficient Path

To understand the effect of the penalty value $\lambda$ (alpha), we will train multiple Ridge models with different alpha values ranging from $0$ (Linear Regression) to $1500$, and plot the values of our coefficients.

- As $\lambda \rightarrow 0$, coefficients remain large and erratic due to multicollinearity.
- As $\lambda$ increases, the coefficients **shrink smoothly** toward zero.
- Ridge shrinks coefficients but **never forces them to exactly $0$**, maintaining all features in the equation.

In [ ]:
# Define range of alpha values (using log scale)
alphas = [0.0, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0]
coef_history = []
val_scores = []
train_scores = []

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    coef_history.append(model.coef_)
    train_scores.append(r2_score(y_train, model.predict(X_train)))
    val_scores.append(r2_score(y_test, model.predict(X_test)))

coef_history = np.array(coef_history)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Coefficient Shrinkage Path
for i in range(coef_history.shape[1]):
    axes[0].plot(alphas, coef_history[:, i], marker='o', linewidth=2, label=f'Feature {i}')
axes[0].set_xscale('symlog', linthresh=0.1)  # symmetric log scale to handle 0
axes[0].set_title('Ridge Coefficients Shrinkage Path as Alpha Increases', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Alpha (Regularization Strength)')
axes[0].set_ylabel('Coefficient Weight')
axes[0].legend(ncol=2)

# Plot 2: Bias-Variance Validation Curve
axes[1].plot(alphas, train_scores, color='#2ecc71', marker='s', linewidth=2.5, label='Train R2 Score')
axes[1].plot(alphas, val_scores, color='#3498db', marker='o', linewidth=2.5, label='Validation R2 Score')
axes[1].set_xscale('symlog', linthresh=0.1)
axes[1].set_title('Bias-Variance Tradeoff: Train vs. Validation R2', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Alpha (Regularization Strength)')
axes[1].set_ylabel('R2 Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"OLS (Alpha=0) R2 Score (Val):      {val_scores[0]:.4f}")
print(f"Best Ridge (Alpha=10) R2 Score (Val): {val_scores[4]:.4f} (Under multicollinearity, Ridge generalized better!)")


## Summary Checklist for Ridge Regression

1. **Scale your Features First:** Because the L2 penalty restricts the size of coefficients, features with larger scales will be penalized disproportionately. Always standardize inputs using `StandardScaler` first.
2. **Tune Alpha ($\lambda$) using Cross-Validation:** The ideal penalty strength varies for every dataset. Utilize `RidgeCV` in Scikit-Learn to perform efficient, built-in search for the optimal alpha.
3. **Do Not Penalize Intercept:** Make sure your custom implementations exclude the bias/intercept term $\beta_0$ from the penalty calculation to maintain correct predictions.
4. **Mitigate Multicollinearity:** If features are highly correlated, standard OLS estimates will have high variance. Use Ridge to stabilize the coefficients and prevent overfitting.